In [22]:

# ============================================================
# Function approximation in orthogonal bases
# ============================================================
#
# This script generates Figure 2 for the paper:
#   Section 4.2 "Function approximation in orthogonal bases"
#
# Purpose
# -------
# The experiment studies approximation of the nonlinear two-dimensional target
#
#     f(x1, x2) = exp(x1 * cos(2 x2)),    (x1, x2) in [-1, 5]^2,
#
# which combines oscillatory and exponential structure and is not well aligned
# with simple low-order polynomial truncations or isotropic kernel feature maps.
#
# The goal is to compare:
#   (i)  Sparse Orthogonal Regression (SOR) in a tensor-product orthonormal
#        Legendre basis,
#   (ii) Ordinary least squares (OLS) in tensor-product monomial and Legendre
#        bases,
#   (iii) Dense nonlinear baselines based on radial basis functions (RBF),
#         random Fourier features (RFF), and gradient boosting (GB).
#
# Experimental setup
# ------------------
# - Samples are drawn uniformly on [-1, 5]^2.
# - The data are split into 20% training and 80% test sets.
# - Additive Gaussian noise is applied to TRAIN labels only.
# - Test RMSE is always evaluated on clean held-out targets.
#
# For polynomial models:
# - SOR uses a fixed tensor-product Legendre dictionary with total degree
#   up to D_MAX, built from orthonormal 1D Legendre functions on [-1, 1].
# - Inputs are mapped from [-1, 5] to [-1, 1] before Legendre evaluation.
# - SOR uses standardized design columns and LASSO, with the regularization
#   parameter selected so that the number of nonzero coefficients is
#   approximately equal to the target complexity k.
# - OLS monomial and OLS Legendre baselines use total-degree truncations with
#   degree d_max chosen so that the number of coefficients is approximately k.
#
# For dense baselines:
# - RBF and RFF use fixed feature banks of size K_MAX.
# - Increasing k means using the first k features of the fixed bank and fitting
#   ridge regression.
# - Gradient boosting uses n_estimators = k as an effective complexity proxy.
#
# Outputs
# -------
# The script saves six figure panels and one summary panel:
#
#   2a.pdf  Test RMSE versus effective parameter count k
#   2b.pdf  SOR coefficient magnitudes for k in {60, 80, 100}
#   2c.pdf  OLS coefficient magnitudes in tensor monomial basis
#   2d.pdf  OLS coefficient magnitudes in tensor Legendre basis
#   2e.pdf  RBF ridge coefficients
#   2f.pdf  RFF ridge coefficients
#
# Connection to the paper
# -----------------------
# This experiment supports the claims made in Section 4.2:
#
# - SOR achieves the lowest test RMSE at matched complexity.
# - In the Legendre basis, SOR identifies stable low-order dominant modes while
#   suppressing unsupported higher-order terms.
# - OLS in monomial coordinates is highly inconsistent across model orders.
# - OLS in orthogonal Legendre coordinates is better scaled and more consistent,
#   but still sensitive to noise at higher orders.
# - RBF and RFF remain dense and highly variable as model size increases.
#
# Notes
# -----
# - For OLS polynomial baselines, parameter count is only approximately matched
#   to k through the selected total-degree truncation.
# - The optional ALPHA_SCALE_BY_K hook is used only to fine-tune sparsity in
#   the SOR coefficient plot if needed; with the current settings it leaves the
#   default target-nnz fitting unchanged.
# - The "Lower Bound" curve in panel 2a is a heuristic reference curve derived
#   from the current target function and should not be interpreted as a direct
#   consequence of Theorem 1.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.linear_model import Lasso
from sklearn.ensemble import GradientBoostingRegressor

# ============================================================
# Output directory
# ============================================================
FIGDIR = "figs"
os.makedirs(FIGDIR, exist_ok=True)

# ============================================================
# CONFIG
# ============================================================
a, b = -1.0, 5.0
N_total = 2000
train_frac = 0.2
noise_std = 0.10          # additive noise std on TRAIN labels (absolute)
seed = 0

D_MAX = 50                # FIXED dictionary for SOR (total degree <= D_MAX)
NZ_TOL = 1e-10

# k-values
K_RMSE = [20, 30, 40, 50, 60, 70, 80, 90, 100]
K_COEF = [60, 80, 100]
K_MAX = max(K_RMSE)

# LASSO solver knobs
LASSO_MAX_ITER = 500000
LASSO_TOL = 1e-5

# RBF / RFF baselines (fixed feature banks)
RIDGE_LAMBDA = 1e-6
PAIR_SAMPLES_FOR_ELL = 6000
ELL_FLOOR = 1e-3

# --- NEW: alpha scaling used ONLY for the SOR coefficient plot (Fig 4c)
#     Set to 1.0 by default. Tweak e.g. 0.8 or 1.2 per k to nudge sparsity.
ALPHA_SCALE_BY_K = {40: 1, 60: 1.0, 80: 1.0}

# ============================================================
# Target function
# ============================================================
def f_target(X):
    X = np.asarray(X, float)
    x1, x2 = X[:, 0], X[:, 1]
    return np.exp(x1*np.cos(2*x2))

# ============================================================
# Tensor-product basis index sets (total degree)
# ============================================================
def multi_indices_exact(total_deg, n_vars):
    if n_vars == 1:
        yield (total_deg,)
        return

    def rec(prefix, remaining_deg, remaining_dims):
        if remaining_dims == 1:
            yield tuple(prefix + (remaining_deg,))
            return
        for a_ in range(remaining_deg + 1):
            yield from rec(prefix + (a_,), remaining_deg - a_, remaining_dims - 1)

    yield from rec((), total_deg, n_vars)

def multi_indices_upto(d_max, n_vars):
    idx = []
    for k in range(d_max + 1):
        idx.extend(list(multi_indices_exact(k, n_vars)))
    return idx

# ============================================================
# Map x in [a,b] to u in [-1,1] for orthogonal bases
# ============================================================
def map_to_minus1_1(x, a, b):
    x = np.asarray(x, float)
    return (2.0 * x - (a + b)) / (b - a)

# ============================================================
# NEW: Orthonormal Legendre "functions"
#   phi_n(u) = sqrt((2n+1)/2) * P_n(u)
# ============================================================
def eval_1d_legendre(u, k):
    u = np.asarray(u, float)
    coeffs = np.zeros(k + 1, float)
    coeffs[-1] = 1.0
    Pk = np.polynomial.legendre.legval(u, coeffs)  # P_k(u)
    return np.sqrt((2.0 * k + 1.0) / 2.0) * Pk      # phi_k(u)

def build_tensor_legendre_design(X, d_max, a, b):
    X = np.asarray(X, float)
    N, n_vars = X.shape
    indexmulti = multi_indices_upto(d_max, n_vars)
    M = len(indexmulti)

    U = map_to_minus1_1(X, a, b)

    vals = []
    for j in range(n_vars):
        uj = U[:, j]
        vals_j = [eval_1d_legendre(uj, k) for k in range(d_max + 1)]
        vals.append(vals_j)

    Phi = np.ones((N, M), float)
    for s, alpha in enumerate(indexmulti):
        col = np.ones(N, float)
        for j, deg in enumerate(alpha):
            col *= vals[j][deg]
        Phi[:, s] = col
    return Phi, indexmulti

# ============================================================
# tensor monomials design (total degree <= d_max)
# ============================================================
def build_tensor_monomial_design(X, d_max):
    """
    Monomials in ORIGINAL coordinates:
      Phi[:,s] = x1^a * x2^b  for (a,b) with a+b <= d_max, in the SAME multi-index ordering
    """
    X = np.asarray(X, float)
    N, n_vars = X.shape
    indexmulti = multi_indices_upto(d_max, n_vars)
    M = len(indexmulti)

    Phi = np.ones((N, M), float)
    for s, alpha in enumerate(indexmulti):
        col = np.ones(N, float)
        for j, deg in enumerate(alpha):
            if deg != 0:
                col *= X[:, j] ** deg
        Phi[:, s] = col
    return Phi, indexmulti

# ============================================================
# Utilities
# ============================================================
def rmse(yhat, y):
    yhat = np.asarray(yhat).reshape(-1)
    y = np.asarray(y).reshape(-1)
    return float(np.sqrt(np.mean((yhat - y) ** 2)))

def count_nnz(coef, tol=NZ_TOL):
    coef = np.asarray(coef).reshape(-1)
    return int(np.count_nonzero(np.abs(coef) > tol))

def standardize_design(Phi):
    Phi = np.asarray(Phi, float)
    col_norms = np.linalg.norm(Phi, axis=0)
    col_norms[col_norms < 1e-12] = 1.0
    return Phi / col_norms, col_norms

def ridge_lstsq(Phi, y, lam):
    Phi = np.asarray(Phi, float)
    y = np.asarray(y, float).reshape(-1)
    p = Phi.shape[1]
    A = np.vstack([Phi, np.sqrt(lam) * np.eye(p)])
    b = np.concatenate([y, np.zeros(p)])
    w, *_ = np.linalg.lstsq(A, b, rcond=None)
    return w

# ============================================================
# OLS helper (fit_intercept=False)
# ============================================================
def ols_lstsq(Phi, y):
    Phi = np.asarray(Phi, float)
    y = np.asarray(y, float).reshape(-1)
    w, *_ = np.linalg.lstsq(Phi, y, rcond=None)
    return w

# ============================================================
# RBF / RFF fixed banks (2D)
# ============================================================
def pairwise_sq_dists(A_, B_):
    A_ = np.asarray(A_, float)
    B_ = np.asarray(B_, float)
    AA = np.sum(A_ * A_, axis=1)[:, None]
    BB = np.sum(B_ * B_, axis=1)[None, :]
    return AA + BB - 2.0 * (A_ @ B_.T)

def estimate_length_scale(X_train, rng):
    X = np.asarray(X_train, float)
    n = X.shape[0]
    m = int(min(PAIR_SAMPLES_FOR_ELL, n * (n - 1) // 2))
    if m < 20:
        return 1.0
    i = rng.integers(0, n, size=m)
    j = rng.integers(0, n, size=m)
    d = np.linalg.norm(X[i] - X[j], axis=1)
    med = float(np.median(d))
    return max(med / np.sqrt(2.0), ELL_FLOOR)

def rbf_features(X, centers, ell):
    d2 = pairwise_sq_dists(X, centers)
    return np.exp(-0.5 * d2 / (ell * ell))

def rff_features(X, omegas, bs):
    proj = X @ omegas.T
    return np.sqrt(2.0 / omegas.shape[0]) * np.cos(proj + bs[None, :])

# ============================================================
# SOR/LASSO: robust alpha selection (UPDATED ONLY HERE previously)
# ============================================================
def alpha_max_for_lasso(Phi_std, y_centered):
    n = Phi_std.shape[0]
    v = np.abs(Phi_std.T @ y_centered)
    return float(np.max(v) / max(n, 1))

def fit_lasso_coef_std(Phi_std, y_centered, alpha):
    m = Lasso(
        alpha=float(alpha),
        fit_intercept=False,
        max_iter=int(LASSO_MAX_ITER),
        tol=float(LASSO_TOL),
        selection="cyclic",
    )
    m.fit(Phi_std, y_centered)
    return np.asarray(m.coef_, float)

def nnz_at_alpha(Phi_std, y_centered, alpha):
    coef_std = fit_lasso_coef_std(Phi_std, y_centered, alpha=alpha)
    return count_nnz(coef_std)

def find_alpha_for_target_nnz(Phi_std, y_centered, k_target, max_decades=30, bisect_steps=30):
    """
    Find alpha so that nnz(coef(alpha)) ~= k_target.
    (same as your current robust bracket+bisection routine)
    """
    amax = alpha_max_for_lasso(Phi_std, y_centered)

    alpha_hi = amax
    nnz_hi = nnz_at_alpha(Phi_std, y_centered, alpha_hi)

    alpha_lo = alpha_hi
    nnz_lo = nnz_hi
    for t in range(max_decades):
        alpha_lo = alpha_hi * (10.0 ** (-(t + 1)))
        nnz_lo = nnz_at_alpha(Phi_std, y_centered, alpha_lo)
        if nnz_lo > 0:
            break

    if nnz_lo == 0:
        alpha_lo = amax * 10.0 ** (-(max_decades + 5))
        nnz_lo = nnz_at_alpha(Phi_std, y_centered, alpha_lo)

    if nnz_lo < k_target:
        alpha_reach = alpha_lo
        nnz_reach = nnz_lo
        for t in range(max_decades):
            alpha_reach = alpha_lo * (10.0 ** (-(t + 1)))
            nnz_reach = nnz_at_alpha(Phi_std, y_centered, alpha_reach)
            if nnz_reach >= k_target:
                alpha_lo = alpha_reach
                nnz_lo = nnz_reach
                break
            alpha_lo = alpha_reach
            nnz_lo = nnz_reach

        if nnz_lo < k_target:
            probe_info = {
                "alpha_max": amax,
                "alpha_hi": alpha_hi,
                "nnz_hi": nnz_hi,
                "alpha_lo": alpha_lo,
                "nnz_lo": nnz_lo,
                "note": "Could not reach k_target; returning smallest alpha tried."
            }
            return float(alpha_lo), probe_info

    if alpha_hi <= alpha_lo:
        alpha_hi, alpha_lo = alpha_lo, alpha_hi
        nnz_hi, nnz_lo = nnz_lo, nnz_hi

    if nnz_hi > k_target:
        for t in range(max_decades):
            alpha_hi = alpha_hi * (10.0 ** (t + 1))
            nnz_hi = nnz_at_alpha(Phi_std, y_centered, alpha_hi)
            if nnz_hi <= k_target:
                break

    log_hi = np.log(alpha_hi)
    log_lo = np.log(alpha_lo)
    best_alpha = alpha_lo
    best_err = abs(nnz_lo - k_target)
    best_nnz = nnz_lo

    for _ in range(bisect_steps):
        log_mid = 0.5 * (log_hi + log_lo)
        alpha_mid = float(np.exp(log_mid))
        nnz_mid = nnz_at_alpha(Phi_std, y_centered, alpha_mid)

        err = abs(nnz_mid - k_target)
        if err < best_err:
            best_err = err
            best_alpha = alpha_mid
            best_nnz = nnz_mid

        if nnz_mid >= k_target:
            log_lo = log_mid
        else:
            log_hi = log_mid

    probe_info = {
        "alpha_max": amax,
        "alpha_hi": alpha_hi,
        "nnz_hi": nnz_hi,
        "alpha_lo": alpha_lo,
        "nnz_lo": nnz_lo,
        "alpha_best": best_alpha,
        "nnz_best": best_nnz,
        "best_err": best_err,
    }
    return float(best_alpha), probe_info

def sor_fit_for_target_k(Phi, y, k_target, alpha_scale=1.0):
    y = np.asarray(y, float).reshape(-1)
    y_mean = float(np.mean(y))
    y_centered = y - y_mean

    Phi_std, col_norms = standardize_design(Phi)

    alpha_hat, probe_info = find_alpha_for_target_nnz(Phi_std, y_centered, k_target=k_target)

    # --- NEW: user-accessible alpha scaling
    alpha_hat = float(alpha_hat) * float(alpha_scale)

    coef_std = fit_lasso_coef_std(Phi_std, y_centered, alpha=alpha_hat)
    coef = coef_std / col_norms
    nnz_final = count_nnz(coef)

    return coef, nnz_final, alpha_hat, y_mean, probe_info

# ============================================================
# pick d_max so that M(d_max) ~ k_target for 2D total-degree basis
# ============================================================
def M_total_degree_2d(d):
    return (d + 1) * (d + 2) // 2

def choose_dmax_for_target_params(k_target, dmax_cap=D_MAX):
    best_d = 0
    best_err = abs(M_total_degree_2d(0) - k_target)
    for d in range(1, dmax_cap + 1):
        err = abs(M_total_degree_2d(d) - k_target)
        if err < best_err:
            best_err = err
            best_d = d
    return best_d

# ============================================================
# Generate fixed train/test data
# ============================================================
rng = np.random.default_rng(seed)
X = rng.uniform(a, b, size=(N_total, 2))

perm = rng.permutation(N_total)
N_train = max(20, int(train_frac * N_total))
train_idx = perm[:N_train]
test_idx = perm[N_train:]

X_train = X[train_idx]
X_test  = X[test_idx]

y_train_clean = f_target(X_train)
y_test_clean  = f_target(X_test)
y_train_noisy = y_train_clean + noise_std * rng.standard_normal(size=y_train_clean.shape)

# ============================================================
# Fixed SOR design (tensor-product Legendre FUNCTIONS, total degree <= D_MAX)
# ============================================================
Phi_all, indexmulti = build_tensor_legendre_design(X, d_max=D_MAX, a=a, b=b)
Phi_train = Phi_all[train_idx]
Phi_test  = Phi_all[test_idx]
M_dict = Phi_train.shape[1]

# ============================================================
# Fixed RBF / RFF banks (size K_MAX)
# ============================================================
ell = estimate_length_scale(X_train, rng)

centers_bank = X_train[rng.choice(X_train.shape[0], size=K_MAX, replace=(X_train.shape[0] < K_MAX))]
Phi_train_rbf_bank = rbf_features(X_train, centers_bank, ell=ell)
Phi_test_rbf_bank  = rbf_features(X_test,  centers_bank, ell=ell)

omegas_bank = rng.normal(0.0, 1.0 / ell, size=(K_MAX, 2))
bs_bank = rng.uniform(0.0, 2.0 * np.pi, size=(K_MAX,))
Phi_train_rff_bank = rff_features(X_train, omegas_bank, bs_bank)
Phi_test_rff_bank  = rff_features(X_test,  omegas_bank, bs_bank)

# ============================================================
# Fit across K_RMSE, store for RMSE + keep weights for K_COEF
# ============================================================
results = {
    "k": [],
    "sor": {"coef": {}, "nnz": [], "alpha": [], "rmse_test": [], "probe": []},
    "rbf": {"w":   {},               "rmse_test": []},
    "rff": {"w":   {},               "rmse_test": []},
    "gb":  {"rmse_test": []},
}

for k in K_RMSE:
    # --- SOR: target nnz ~ k
    coef_sor, nnz_sor, alpha_hat, ymean_sor, probe_info = sor_fit_for_target_k(
        Phi_train, y_train_noisy, k_target=k
    )
    yhat_te_sor = ymean_sor + Phi_test @ coef_sor

    # --- NEW: print immediately alpha and nnz as code runs
    print(f"[SOR] target k={k:>3d} -> nnz={nnz_sor:>3d}, alpha={alpha_hat:.3e}")

    # --- RBF: ridge on first k features (fixed bank prefix)
    Phi_te_rbf = Phi_test_rbf_bank[:, :k]
    Phi_tr_rbf = Phi_train_rbf_bank[:, :k]
    w_rbf = ridge_lstsq(Phi_tr_rbf, y_train_noisy, lam=RIDGE_LAMBDA)
    yhat_te_rbf = Phi_te_rbf @ w_rbf

    # --- RFF: ridge on first k features (fixed bank prefix)
    Phi_te_rff = Phi_test_rff_bank[:, :k]
    Phi_tr_rff = Phi_train_rff_bank[:, :k]
    w_rff = ridge_lstsq(Phi_tr_rff, y_train_noisy, lam=RIDGE_LAMBDA)
    yhat_te_rff = Phi_te_rff @ w_rff

    # --- Gradient Boosting: treat "k" as number of trees
    gb = GradientBoostingRegressor(
        n_estimators=int(k),
        learning_rate=0.1,
        max_depth=3,
        random_state=seed,
        subsample=1.0,
    )
    gb.fit(X_train, y_train_noisy)
    yhat_te_gb = gb.predict(X_test)

    # store
    results["k"].append(k)

    results["sor"]["nnz"].append(nnz_sor)
    results["sor"]["alpha"].append(alpha_hat)
    results["sor"]["probe"].append(probe_info)
    results["sor"]["rmse_test"].append(rmse(yhat_te_sor, y_test_clean))
    if k in K_COEF:
        results["sor"]["coef"][k] = coef_sor

    results["rbf"]["rmse_test"].append(rmse(yhat_te_rbf, y_test_clean))
    if k in K_COEF:
        results["rbf"]["w"][k] = w_rbf

    results["rff"]["rmse_test"].append(rmse(yhat_te_rff, y_test_clean))
    if k in K_COEF:
        results["rff"]["w"][k] = w_rff

    results["gb"]["rmse_test"].append(rmse(yhat_te_gb, y_test_clean))

ks = np.array(results["k"], dtype=int)

# ============================================================
# Global plotting style for all figures
# ============================================================
PLOT_STYLE = {
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "axes.linewidth": 1.1,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "savefig.format": "pdf",
}
plt.rcParams.update(PLOT_STYLE)

AXIS_LABEL_SIZE = PLOT_STYLE["axes.labelsize"]
TITLE_SIZE = PLOT_STYLE["axes.titlesize"]
TICK_LABEL_SIZE = PLOT_STYLE["xtick.labelsize"]
LEGEND_SIZE = PLOT_STYLE["legend.fontsize"]

# Match RBF/RFF appearance across all coefficient figures
COEFF_LINEWIDTH = 1.8
COEFF_MARKERSIZE = 4

def apply_common_axis_style(ax):
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=TICK_LABEL_SIZE,
        length=PLOT_STYLE["xtick.major.size"],
        width=PLOT_STYLE["xtick.major.width"],
    )
    ax.xaxis.label.set_size(AXIS_LABEL_SIZE)
    ax.yaxis.label.set_size(AXIS_LABEL_SIZE)
    ax.title.set_size(TITLE_SIZE)

# ============================================================
# Shared styling for coefficient plots
# ============================================================
color_by_k = {60: "C0", 80: "C1", 100: "C2"}
draw_order = [100, 80, 60]   # higher order on bottom, lower on top

# ============================================================
# FIG 2(a): RBF weights
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
for k in K_COEF:
    w = results["rbf"]["w"][k]
    mags = np.abs(w)
    x = np.arange(1, mags.size + 1)
    ax.plot(
        x, mags,
        marker="o",
        linewidth=COEFF_LINEWIDTH,
        markersize=COEFF_MARKERSIZE,
        label=rf"RBF $k={k}$"
    )

ax.set_xscale("log")
ax.set_xlabel("Feature Index + 1 (Fixed RBF Bank Prefix)")
ax.set_ylabel(r"Coefficient Magnitude $|w_m|$")
ax.set_title(rf"RBF Ridge Weights (Fixed Bank, $\ell={ell:.3g}$, $\lambda={RIDGE_LAMBDA:g}$)")
ax.grid(True, which="both", alpha=0.25)
ax.legend(fontsize=LEGEND_SIZE)
apply_common_axis_style(ax)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "2e.pdf"), bbox_inches="tight")
plt.close(fig)

# ============================================================
# FIG 2(b): RFF weights
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
for k in K_COEF:
    w = results["rff"]["w"][k]
    mags = np.abs(w)
    x = np.arange(1, mags.size + 1)
    ax.plot(
        x, mags,
        marker="o",
        linewidth=COEFF_LINEWIDTH,
        markersize=COEFF_MARKERSIZE,
        label=rf"RFF $k={k}$"
    )

ax.set_xscale("log")
ax.set_xlabel("Feature Index + 1 (Fixed RFF Bank Prefix)")
ax.set_ylabel(r"Coefficient Magnitude $|w_m|$")
ax.set_title(rf"RFF Ridge Weights (Fixed Bank, $\lambda={RIDGE_LAMBDA:g}$)")
ax.grid(True, which="both", alpha=0.25)
ax.legend(fontsize=LEGEND_SIZE)
apply_common_axis_style(ax)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "2f.pdf"), bbox_inches="tight")
plt.close(fig)

# ============================================================
# FIG 2(e): SOR coefficient magnitudes (nonzero only)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
handles = {}

for k_target in draw_order:
    alpha_scale = float(ALPHA_SCALE_BY_K.get(k_target, 1.0))

    coef_k, nnz_k, alpha_k, ymean_k, _ = sor_fit_for_target_k(
        Phi_train, y_train_noisy, k_target=k_target, alpha_scale=alpha_scale
    )

    mags = np.abs(coef_k)
    nz_idx = np.where(mags > NZ_TOL)[0]
    if nz_idx.size == 0:
        continue

    nz_mags = mags[nz_idx]
    if nz_idx.size >= k_target:
        take = np.argsort(-nz_mags)[:k_target]
        sel_idx = np.sort(nz_idx[take])   # basis order
        yplot = mags[sel_idx]
        xplot = np.arange(1, k_target + 1)
    else:
        sel_idx = np.sort(nz_idx)
        yplot = mags[sel_idx]
        yplot = np.concatenate([yplot, np.full(k_target - yplot.size, np.nan)])
        xplot = np.arange(1, k_target + 1)

    ln, = ax.plot(
        xplot, yplot,
        marker="o",
        linewidth=COEFF_LINEWIDTH,
        markersize=COEFF_MARKERSIZE,
        color=color_by_k[k_target],
        zorder=10 + (max(K_COEF) - k_target),
        label=rf"$k={k_target}$"
    )
    handles[k_target] = ln

ax.set_xscale("log")
ax.set_xlim(1, max(K_COEF))
ax.set_xlabel("Nonzero Coefficient Number (Basis Order)")
ax.set_ylabel(r"Coefficient Magnitude $|c|$")
ax.set_title("SOR (Sparse + Orthogonal): Coefficient Magnitudes (Nonzero Only)")
ax.grid(True, which="both", alpha=0.3)

legend_handles = [handles[k] for k in K_COEF if k in handles]
legend_labels = [rf"$k={k}$" for k in K_COEF if k in handles]
ax.legend(legend_handles, legend_labels, fontsize=LEGEND_SIZE)

apply_common_axis_style(ax)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "2b.pdf"), bbox_inches="tight")
plt.close(fig)

# ============================================================
# FIG 2(d): OLS coefficients (orthogonal / tensor Legendre)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
handles = {}

for k_target in draw_order:
    dmax = choose_dmax_for_target_params(k_target, dmax_cap=D_MAX)
    Phi_all_d, _ = build_tensor_legendre_design(X, d_max=dmax, a=a, b=b)
    Phi_tr_d = Phi_all_d[train_idx]

    coef_ols = ols_lstsq(Phi_tr_d, y_train_noisy)
    mags = np.abs(coef_ols)

    M = mags.size
    xplot = np.arange(1, M + 1)
    yplot = mags

    ln, = ax.plot(
        xplot, yplot,
        marker="o",
        linewidth=COEFF_LINEWIDTH,
        markersize=COEFF_MARKERSIZE,
        color=color_by_k[k_target],
        zorder=10 + (max(K_COEF) - k_target),
        label=rf"$k\approx{k_target}$ ($d_{{\max}}={dmax}$, $M={M}$)"
    )
    handles[k_target] = ln

ax.set_xscale("log")
ax.set_xlim(1, max(K_COEF))
ax.set_xlabel("Coefficient Number (Basis Order)")
ax.set_ylabel(r"Coefficient Magnitude $|c|$")
ax.set_title("OLS Coefficients (Orthogonal / Tensor Legendre): Magnitudes (Basis Order)")
ax.grid(True, which="both", alpha=0.3)

legend_handles = [handles[k] for k in K_COEF if k in handles]
legend_labels = [rf"$k={k}$" for k in K_COEF if k in handles]
ax.legend(legend_handles, legend_labels, fontsize=LEGEND_SIZE)

apply_common_axis_style(ax)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "2d.pdf"), bbox_inches="tight")
plt.close(fig)

# ============================================================
# FIG 2(c): OLS coefficients (monomials / tensor monomials)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
handles = {}

for k_target in draw_order:
    dmax = choose_dmax_for_target_params(k_target, dmax_cap=D_MAX)
    Phi_all_d, _ = build_tensor_monomial_design(X, d_max=dmax)
    Phi_tr_d = Phi_all_d[train_idx]

    coef_ols = ols_lstsq(Phi_tr_d, y_train_noisy)
    mags = np.abs(coef_ols)

    M = mags.size
    xplot = np.arange(1, M + 1)
    yplot = mags

    ln, = ax.plot(
        xplot, yplot,
        marker="o",
        linewidth=COEFF_LINEWIDTH,
        markersize=COEFF_MARKERSIZE,
        color=color_by_k[k_target],
        zorder=10 + (max(K_COEF) - k_target),
        label=rf"$k\approx{k_target}$ ($d_{{\max}}={dmax}$, $M={M}$)"
    )
    handles[k_target] = ln

ax.set_xscale("log")
ax.set_xlim(1, max(K_COEF))
ax.set_xlabel("Coefficient Number (Basis Order)")
ax.set_ylabel(r"Coefficient Magnitude $|c|$")
ax.set_title("OLS Coefficients (Monomials / Tensor Monomials): Magnitudes (Basis Order)")
ax.grid(True, which="both", alpha=0.3)

legend_handles = [handles[k] for k in K_COEF if k in handles]
legend_labels = [rf"$k={k}$" for k in K_COEF if k in handles]
ax.legend(legend_handles, legend_labels, fontsize=LEGEND_SIZE)

apply_common_axis_style(ax)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "2c.pdf"), bbox_inches="tight")
plt.close(fig)

[SOR] target k= 20 -> nnz= 20, alpha=7.178e-02
[SOR] target k= 30 -> nnz= 30, alpha=5.530e-02
[SOR] target k= 40 -> nnz= 40, alpha=4.724e-02
[SOR] target k= 50 -> nnz= 50, alpha=4.184e-02
[SOR] target k= 60 -> nnz= 60, alpha=3.195e-02
[SOR] target k= 70 -> nnz= 70, alpha=2.903e-02
[SOR] target k= 80 -> nnz= 80, alpha=2.229e-02
[SOR] target k= 90 -> nnz= 90, alpha=2.019e-02
[SOR] target k=100 -> nnz=100, alpha=1.846e-02


In [23]:
# ============================================================
# Global plotting style for all figures
# ============================================================

RMSE_LINEWIDTH = 2.0
RMSE_MARKERSIZE = 5

def apply_common_axis_style(ax):
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=TICK_LABEL_SIZE,
        length=PLOT_STYLE["xtick.major.size"],
        width=PLOT_STYLE["xtick.major.width"],
    )
    ax.xaxis.label.set_size(AXIS_LABEL_SIZE)
    ax.yaxis.label.set_size(AXIS_LABEL_SIZE)
    ax.title.set_size(TITLE_SIZE)

# ============================================================
# FIG 2: Test RMSE vs k (SOR/RBF/RFF/GB + OLS orth/mono)
# ============================================================

# --- compute OLS RMSE curves (orthogonal Legendre + monomials) aligned with ks ---
ols_leg_rmse = []
ols_mono_rmse = []

for k in ks:
    # pick dmax so M(dmax) ~ k (2D total-degree basis)
    dmax_k = choose_dmax_for_target_params(int(k), dmax_cap=D_MAX)

    # --- OLS on tensor Legendre (orthogonal basis)
    Phi_all_leg_k, _ = build_tensor_legendre_design(X, d_max=dmax_k, a=a, b=b)
    Phi_tr_leg_k = Phi_all_leg_k[train_idx]
    Phi_te_leg_k = Phi_all_leg_k[test_idx]
    c_leg = ols_lstsq(Phi_tr_leg_k, y_train_noisy)
    yhat_leg = Phi_te_leg_k @ c_leg
    ols_leg_rmse.append(rmse(yhat_leg, y_test_clean))

    # --- OLS on tensor monomials
    Phi_all_mono_k, _ = build_tensor_monomial_design(X, d_max=dmax_k)
    Phi_tr_mono_k = Phi_all_mono_k[train_idx]
    Phi_te_mono_k = Phi_all_mono_k[test_idx]
    c_mono = ols_lstsq(Phi_tr_mono_k, y_train_noisy)
    yhat_mono = Phi_te_mono_k @ c_mono
    ols_mono_rmse.append(rmse(yhat_mono, y_test_clean))

ols_leg_rmse = np.asarray(ols_leg_rmse, float)
ols_mono_rmse = np.asarray(ols_mono_rmse, float)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    ks, results["sor"]["rmse_test"],
    marker="o", linewidth=RMSE_LINEWIDTH, markersize=RMSE_MARKERSIZE,
    label="SOR"
)
ax.plot(
    ks, results["rbf"]["rmse_test"],
    marker="s", linewidth=RMSE_LINEWIDTH, markersize=RMSE_MARKERSIZE,
    label="RBF"
)
ax.plot(
    ks, results["rff"]["rmse_test"],
    marker="^", linewidth=RMSE_LINEWIDTH, markersize=RMSE_MARKERSIZE,
    label="RFF"
)
ax.plot(
    ks, results["gb"]["rmse_test"],
    marker="d", linewidth=RMSE_LINEWIDTH, markersize=RMSE_MARKERSIZE,
    label="GB"
)

# --- OLS curves
ax.plot(
    ks, ols_leg_rmse,
    marker="v", linewidth=RMSE_LINEWIDTH, markersize=RMSE_MARKERSIZE,
    label="OLS (Legendre)"
)
ax.plot(
    ks, ols_mono_rmse,
    marker="x", linewidth=RMSE_LINEWIDTH, markersize=RMSE_MARKERSIZE,
    label="OLS (Monomials)"
)

# --- Estimated lower bound line (uses actual nnz, not target k)
N_eff = np.maximum(1, np.asarray(results["sor"]["nnz"], dtype=float))

# NOTE: If you change f_target, update grad_norm and fvals accordingly.
# Current target:
#   f(x1, x2) = exp(x1 * cos(2 x2))
# so
#   ||grad f|| = exp(x1*cos(2x2)) * sqrt(cos^2(2x2) + 4*x1^2*sin^2(2x2))

x1 = X_test[:, 0]
x2 = X_test[:, 1]

fvals = np.exp(x1 * np.cos(2.0 * x2))
grad_norm = fvals * np.sqrt(
    np.cos(2.0 * x2) ** 2 + 4.0 * x1**2 * np.sin(2.0 * x2) ** 2
)

vol = (b - a) ** 2
Var_est = vol * float(np.mean(np.abs(grad_norm)))

scale = float(np.sqrt(np.mean(fvals**2)))
C_norm = Var_est / scale

ax.plot(
    ks, C_norm / np.sqrt(np.pi * N_eff),
    linewidth=RMSE_LINEWIDTH,
    label="Est. Lower Bound"
)

ax.set_xlabel(r"Parameter Count $k$")
ax.set_ylabel("Test RMSE")
ax.set_title(
    rf"Test RMSE vs. $k$ on $[{a:g},{b:g}]^2$ "
    rf"(Train={train_frac*100:.0f}%, Noise $\sigma = ${noise_std:g}, $D_{{\max}}={D_MAX}$)"
)
ax.grid(True, alpha=0.25)

ax.legend(
    fontsize=LEGEND_SIZE,
    loc="lower left",
    bbox_to_anchor=(0.0, 0.24),
    ncol=1,
    frameon=True,
    borderpad=0.3,
    labelspacing=0.3,
    handlelength=1.8,
    handletextpad=0.5,
)

apply_common_axis_style(ax)
fig.tight_layout()

out_rmse = os.path.join(FIGDIR, "2a.pdf")
fig.savefig(out_rmse, bbox_inches="tight")
plt.close(fig)

# ============================================================
# Console diagnostics (robust to both probe formats)
# ============================================================
print("Saved figures to:", FIGDIR)

print("\nSOR nnz/alpha diagnostics:")
for k, nnz_k, a_k, probe in zip(
    ks, results["sor"]["nnz"], results["sor"]["alpha"], results["sor"]["probe"]
):
    # Old (3-probe) format
    if isinstance(probe, dict) and ("alphas" in probe) and ("nnz_probe" in probe):
        alphas = probe["alphas"]
        nnzp = probe["nnz_probe"]
        amax = probe.get("alpha_max", np.nan)
        print(f"  target k={int(k):>3d} -> nnz={int(nnz_k):>3d}, alpha_hat={a_k:.3e}, alpha_max={amax:.3e}, probes:")
        for ap, np_ in zip(alphas, nnzp):
            print(f"      alpha={float(ap):.3e} -> nnz={int(np_):>4d}")

    # New (bracket/bisect) format
    elif isinstance(probe, dict) and ("alpha_hi" in probe) and ("alpha_lo" in probe):
        amax = probe.get("alpha_max", np.nan)
        alpha_hi = probe.get("alpha_hi", np.nan)
        nnz_hi = probe.get("nnz_hi", np.nan)
        alpha_lo = probe.get("alpha_lo", np.nan)
        nnz_lo = probe.get("nnz_lo", np.nan)
        alpha_best = probe.get("alpha_best", a_k)
        nnz_best = probe.get("nnz_best", nnz_k)
        note = probe.get("note", "")
        print(f"  target k={int(k):>3d} -> nnz={int(nnz_k):>3d}, alpha_hat={a_k:.3e}, alpha_max={amax:.3e}")
        print(
            f"      bracket: alpha_hi={float(alpha_hi):.3e} "
            f"(nnz={int(nnz_hi) if np.isfinite(nnz_hi) else nnz_hi}), "
            f"alpha_lo={float(alpha_lo):.3e} "
            f"(nnz={int(nnz_lo) if np.isfinite(nnz_lo) else nnz_lo})"
        )
        print(
            f"      best:    alpha_best={float(alpha_best):.3e} "
            f"(nnz_best={int(nnz_best) if np.isfinite(nnz_best) else nnz_best})"
            + (f"  NOTE: {note}" if note else "")
        )

    else:
        print(
            f"  target k={int(k):>3d} -> nnz={int(nnz_k):>3d}, "
            f"alpha_hat={a_k:.3e}, "
            f"probe_info keys={list(probe.keys()) if isinstance(probe, dict) else type(probe)}"
        )

Saved figures to: figs

SOR nnz/alpha diagnostics:
  target k= 20 -> nnz= 20, alpha_hat=7.178e-02, alpha_max=2.270e-01
      bracket: alpha_hi=2.270e-01 (nnz=0), alpha_lo=2.270e-02 (nnz=78)
      best:    alpha_best=7.178e-02 (nnz_best=20)
  target k= 30 -> nnz= 30, alpha_hat=5.530e-02, alpha_max=2.270e-01
      bracket: alpha_hi=2.270e-01 (nnz=0), alpha_lo=2.270e-02 (nnz=78)
      best:    alpha_best=5.530e-02 (nnz_best=30)
  target k= 40 -> nnz= 40, alpha_hat=4.724e-02, alpha_max=2.270e-01
      bracket: alpha_hi=2.270e-01 (nnz=0), alpha_lo=2.270e-02 (nnz=78)
      best:    alpha_best=4.724e-02 (nnz_best=40)
  target k= 50 -> nnz= 50, alpha_hat=4.184e-02, alpha_max=2.270e-01
      bracket: alpha_hi=2.270e-01 (nnz=0), alpha_lo=2.270e-02 (nnz=78)
      best:    alpha_best=4.184e-02 (nnz_best=50)
  target k= 60 -> nnz= 60, alpha_hat=3.195e-02, alpha_max=2.270e-01
      bracket: alpha_hi=2.270e-01 (nnz=0), alpha_lo=2.270e-02 (nnz=78)
      best:    alpha_best=3.195e-02 (nnz_best=60)
  ta